# Aula 2 - Context Engineering

## Versão 2.0 do projeto: um assistente que seleciona contexto

Nesta aula, a aplicação vai selecionar documentos locais antes de chamar uma LLM. Precisamos responder: "qual informação entra na requisição para responder esta pergunta?"

Ao final, você deve conseguir:

- separar instruções, pergunta, histórico e dados externos;
- selecionar documentos por uma regra simples de relevância;
- limitar o tamanho do contexto;
- montar uma requisição que identifica suas fontes;
- comparar uma resposta sem contexto com uma resposta contextualizada.

## 1. Configuração do ambiente

Instale as dependências pelo terminal, seguindo o `README.md` desta pasta. Copie `env.example` para `.env` e preencha `GROQ_API_KEY`. Nunca coloque a chave diretamente neste arquivo nem faça commit do `.env`.

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise RuntimeError(
        "GROQ_API_KEY não encontrada. Copie env.example para .env e informe sua chave."
    )

client = OpenAI(
    api_key=api_key,
    base_url=os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),
)

MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")
print(f"Modelo configurado: {MODEL}")

Modelo configurado: openai/gpt-oss-120b


## 2. Crie uma base local de documentos

Cada documento tem um identificador, um título, um conteúdo e termos que ajudam a explicar por que ele foi selecionado. Em um sistema real, essa base poderia vir de arquivos, banco de dados ou uma API.

In [3]:
# TODO: crie uma lista chamada `knowledge_base` com pelo menos 5 documentos.
# Cada item deve ter: id, title, content e keywords.

knowledge_base = [
    {
        "id": "doc-01",
        "title": "Normalização de Banco de Dados Relacional",
        "keywords": ["banco de dados", "sql", "normalização", "1fn", "2fn", "3fn", "tabelas"],
        "content": (
            "A normalização em bancos relacionais organiza os dados para reduzir redundâncias e anomalias de atualização. "
            "A Primeira Forma Normal (1FN) exige atomicidade dos atributos. A Segunda Forma Normal (2FN) elimina dependências parciais "
            "da chave primária. A Terceira Forma Normal (3FN) remove dependências transitivas entre atributos não-chave."
        ),
    },
    {
        "id": "doc-02",
        "title": "Complexidade de Algoritmos e Notação Big-O",
        "keywords": ["algoritmos", "complexidade", "big-o", "ordenação", "busca", "tempo"],
        "content": (
            "A Notação Big-O descreve o comportamento assintótico de um algoritmo conforme a entrada cresce. "
            "A busca binária opera em tempo logarítmico O(log n) sobre arranjos ordenados. Algoritmos eficientes de ordenação, "
            "como o MergeSort e o QuickSort, possuem custo médio de tempo O(n log n)."
        ),
    },
    {
        "id": "doc-03",
        "title": "Modelo OSI e Camadas de Rede",
        "keywords": ["redes", "osi", "tcp/ip", "transporte", "roteamento", "pacotes"],
        "content": (
            "O modelo OSI divide a comunicação de rede em sete camadas: Física, Enlace, Rede, Transporte, Sessão, Apresentação e Aplicação. "
            "O protocolo TCP opera na camada de transporte garantindo entrega ordenada e controle de fluxo, enquanto o protocolo IP "
            "atua na camada de rede realizando o roteamento de pacotes entre hosts."
        ),
    },
    {
        "id": "doc-04",
        "title": "Gerenciamento de Processos e Deadlocks",
        "keywords": ["sistemas operacionais", "processos", "threads", "deadlock", "concorrência", "recursos"],
        "content": (
            "Um deadlock em sistemas operacionais ocorre quando processos entram em espera mútua indefinida por recursos. "
            "Para ocorrer deadlock, quatro condições de Coffman devem coexistir: exclusão mútua, posse e espera, não preempção "
            "e espera circular. Mecanismos como o algoritmo do banqueiro evitam esse estado."
        ),
    },
    {
        "id": "doc-05",
        "title": "Padrões de Projeto: Singleton e Factory Method",
        "keywords": ["engenharia de software", "design patterns", "orientação a objetos", "singleton", "factory"],
        "content": (
            "Padrões de projeto criacionais definem mecanismos flexíveis de instanciação. O padrão Singleton garante uma única instância "
            "global de uma classe durante todo o ciclo de vida da aplicação. O Factory Method delega a responsabilidade de criação "
            "de objetos concretos para subclasses através de interfaces compartilhadas."
        ),
    },
]
    
# Inclua documentos sobre disciplinas de sua preferência.

## 3. Selecione o contexto

Vamos usar uma estratégia transparente: normalizar os termos da pergunta e contar quantos deles aparecem em cada documento. Isso é suficiente para enxergar a etapa de seleção, mas não entende bem sinônimos.

In [7]:
import re

STOPWORDS = {
    "a", "o", "e", "de", "da", "do", "das", "dos", "em",
    "um", "uma", "para", "por", "com", "que", "na", "no",
    "as", "os", "qual", "quais", "como", "é", "são",
}

# Normalizamos o prompt do usuário, simulando a vetorização.
def normalize_terms(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    return {term for term in text.split() if len(term) > 2 and term not in STOPWORDS}

# Ajudamos o agente a retornar com o contexto que mais possui os termos do prompt, simulando o retrieval.
def select_context(query, documents, max_docs=3, max_chars=1800):
    query_terms = normalize_terms(query)
    ranked = []

    for document in documents:
        searchable = " ".join([
            document["title"],
            document["content"],
            " ".join(document.get("keywords", [])),
        ])
        score = len(query_terms & normalize_terms(searchable))
        if score > 0:
            ranked.append({**document, "score": score})

    ranked.sort(key=lambda document: (-document["score"], document["id"]))

    selected = []
    used_chars = 0
    for document in ranked:
        cost = len(document["title"]) + len(document["content"])
        if selected and used_chars + cost > max_chars:
            continue
        selected.append(document)
        used_chars += cost
        if len(selected) >= max_docs:
            break

    return selected

## 4. Formate o contexto com fontes

O modelo precisa distinguir os dados externos da pergunta. Também queremos preservar as fontes para que a resposta consiga apontar de onde veio a informação.

In [8]:
# O bloco contém ids de fonte para facilitar inspeção e atribuição. O modelo recebe dados externos claramente separados da pergunta.
def build_context(selected_documents):
    if not selected_documents:
        return "Nenhum documento relevante foi selecionado."

    blocks = ["<contexto>"]
    for document in selected_documents:
        blocks.append(
            f"[fonte: {document['id']} | relevância: {document['score']}]\n"
            f"{document['title']}\n{document['content']}"
        )
    blocks.append("</contexto>")
    return "\n\n".join(blocks)

## 5. Veja o que entrou no contexto

Antes de chamar a LLM, inspecione a decisão da aplicação. Context Engineering também é observabilidade: precisamos conseguir explicar qual informação foi enviada.

In [9]:
query ="O que é deadlock e como funciona a normalização em banco de dados?"
selected = select_context(query, knowledge_base)
context = build_context(selected)

print(context)
print(f"\nDocumentos selecionados: {len(selected)}")
print(f"Caracteres de contexto: {len(context)}")

<contexto>

[fonte: doc-01 | relevância: 3]
Normalização de Banco de Dados Relacional
A normalização em bancos relacionais organiza os dados para reduzir redundâncias e anomalias de atualização. A Primeira Forma Normal (1FN) exige atomicidade dos atributos. A Segunda Forma Normal (2FN) elimina dependências parciais da chave primária. A Terceira Forma Normal (3FN) remove dependências transitivas entre atributos não-chave.

[fonte: doc-04 | relevância: 1]
Gerenciamento de Processos e Deadlocks
Um deadlock em sistemas operacionais ocorre quando processos entram em espera mútua indefinida por recursos. Para ocorrer deadlock, quatro condições de Coffman devem coexistir: exclusão mútua, posse e espera, não preempção e espera circular. Mecanismos como o algoritmo do banqueiro evitam esse estado.

</contexto>

Documentos selecionados: 2
Caracteres de contexto: 812


## 6. Monte a requisição com contexto

A instrução do sistema define como tratar as fontes. A mensagem do usuário contém o contexto selecionado e a pergunta atual. A aplicação continua responsável por montar tudo antes da chamada.

In [10]:
# TODO: crie `messages_com_contexto` com uma mensagem system e uma user.
# Instrua o assistente a usar apenas o contexto fornecido, citar os ids
# das fontes quando possível e dizer quando o contexto não for suficiente.
# Envie a lista para o modelo e salve a resposta em `response_com_contexto`.
messages_com_contexto =[
{
        "role": "system",
        "content": (
            "Você é um assistente de estudos. Use apenas as informações dentro de <contexto>. "
            "Cite os ids das fontes quando fizer uma afirmação baseada nelas. "
            "Se o contexto não for suficiente, diga isso claramente."
        ),
    },
    {
        "role": "user",
        "content": f"{context}\n\nPergunta: {query}",
    },
]

response_com_contexto = client.chat.completions.create(
    model=MODEL,
    messages=messages_com_contexto,
    temperature = 0,
)

In [11]:
# TODO: mostre o texto de `response_com_contexto` e os ids das fontes selecionadas.

print(response_com_contexto.choices[0].message.content)
print(f"\nFontes enviadas: {[document['id'] for document in selected]}")

**Deadlock**

Um deadlock (ou impasse) ocorre em sistemas operacionais quando dois ou mais processos ficam em espera mútua indefinida por recursos que precisam para continuar a execução. Para que um deadlock se manifeste, as quatro condições de Coffman precisam estar presentes simultaneamente:  

1. **Exclusão mútua** – um recurso só pode ser usado por um processo de cada vez;  
2. **Posse e espera** – um processo já possui recursos e continua a solicitar outros;  
3. **Não preempção** – os recursos não podem ser retirados à força de um processo;  
4. **Espera circular** – existe uma cadeia de processos onde cada um espera por um recurso que o próximo processo possui【doc-04】.

**Normalização de Banco de Dados Relacional**

A normalização organiza os dados em um banco relacional com o objetivo de reduzir redundâncias e evitar anomalias de atualização. Ela é estruturada em diferentes “formas normais”:

| Forma Normal | O que exige |
|--------------|-------------|
| **1ª Forma Normal (1FN

In [ ]:
# Exemplo de instruções para o assistente:
# messages_com_contexto = [
#     {
#         "role": "system",
#         "content": (
#             "Você é um assistente de estudos. Use apenas as informações dentro de <contexto>. "
#             "Cite os ids das fontes quando fizer uma afirmação baseada nelas. "
#             "Se o contexto não for suficiente, diga isso claramente."
#         ),
#     },
#     {
#         "role": "user",
#         "content": f"{context}\n\nPergunta: {query}",
#     },
# ]

## 7. Compare sem contexto e com contexto

Escolha uma pergunta que dependa de uma informação da `knowledge_base`. Faça duas chamadas independentes. A primeira recebe apenas a pergunta; a segunda recebe o bloco selecionado. Compare especificidade, fontes e grau de confiança.

In [12]:
# TODO: faça uma chamada sem contexto e salve em `response_sem_contexto`.
# Depois faça uma chamada com o contexto selecionado e salve em
# `response_com_contexto_2`. Imprima as duas respostas lado a lado.

response_sem_contexto = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": query}
    ],
    temperature=0,
)


messages_com_contexto_2 = [
    {
        "role": "system",
        "content": (
            "Você é um assistente de estudos. Use apenas as informações dentro de <contexto>. "
            "Cite os ids das fontes quando fizer uma afirmação baseada nelas. "
            "Se o contexto não for suficiente, diga isso claramente."
        ),
    },
    {
        "role": "user",
        "content": f"{context}\n\nPergunta: {query}",
    },
]

response_com_contexto_2 = client.chat.completions.create(
    model=MODEL,
    messages=messages_com_contexto_2,
    temperature=0,
)


print("=== RESPOSTA SEM CONTEXTO ===")
print(response_sem_contexto.choices[0].message.content)

print("\n" + "=" * 50 + "\n")

print("=== RESPOSTA COM CONTEXTO (GROUNDED) ===")
print(response_com_contexto_2.choices[0].message.content)

=== RESPOSTA SEM CONTEXTO ===
**Deadlock** e **normalização de banco de dados** são conceitos fundamentais, mas pertencem a áreas diferentes: o primeiro está relacionado à concorrência e ao controle de acesso a recursos; o segundo trata da organização lógica dos dados. Vamos entender cada um deles separadamente e, ao final, ver como eles podem interagir em um sistema real.

---

## 1️⃣ O que é **deadlock**?

### Definição
Deadlock (ou impasse) ocorre quando **dois ou mais processos/threads ficam bloqueados indefinidamente porque cada um está esperando que o outro libere um recurso que ele precisa** para continuar a execução.

### Como isso acontece?
Imagine que temos duas transações (T1 e T2) e dois recursos (R1 e R2). O cenário clássico de deadlock segue esta sequência:

| Passo | Transação | Ação |
|------|-----------|------|
| 1 | T1 | **Bloqueia** R1 |
| 2 | T2 | **Bloqueia** R2 |
| 3 | T1 | **Tenta bloquear** R2 → **fica esperando** (R2 já está com T2) |
| 4 | T2 | **Tenta bloquea

## 8. Experimente o orçamento de contexto

Altere `max_chars` na função de seleção. Observe o que desaparece quando o orçamento diminui e por que a aplicação precisa escolher uma política explícita.

In [13]:
for budget in [500, 1800]:
    selected_for_budget = select_context(
        query, knowledge_base, max_docs=3, max_chars=budget
    )
    context_for_budget = build_context(selected_for_budget)
    print(f"orçamento={budget}: ids={[document['id'] for document in selected_for_budget]} | caracteres={len(context_for_budget)}")

orçamento=500: ids=['doc-01'] | caracteres=437
orçamento=1800: ids=['doc-01', 'doc-04'] | caracteres=812


## 9. Checkpoint da V2

A V2 está pronta quando:

- a base de documentos existe;
- a seleção ocorre antes da chamada;
- o limite de documentos e caracteres é explícito;
- o contexto preserva as fontes;
- você comparou respostas sem contexto e com contexto;
- a chave continua no `.env`;
- as alterações foram salvas em um commit separado.
